# ANLP — Session 4: LLMs for NLP

**Hands-on notebook**

In this notebook, we'll run the **same three tasks** as in session 3 (sentiment, keywords, NER) — but this time by **prompting an LLM** instead of training or fine-tuning a dedicated model.

The core question for each task: does the LLM match the dedicated model, and at what cost (latency, price, reproducibility)?

**By the end of this notebook, you should be able to:**
- Serve and query a local LLM through an OpenAI-compatible API (via Ollama)
- Build reusable prompt templates in Python
- Compare zero-shot LLM prompting to the dedicated models from session 3
- Force an LLM to return structured (schema-validated) output instead of free text
- Compare zero-shot vs. few-shot prompting
- Reason about the practical trade-offs (cost, latency, reproducibility, privacy, context length) of using an LLM for classic NLP tasks


## Setup

To run LLMs locally, we'll use [Ollama](https://ollama.com/).

Ollama is a free, open-source platform that allows you to download, manage, and run Large Language Models (LLMs) such as Qwen, Gemma, and Mistral directly on your own computer.

The Ollama server exposes an OpenAI-compatible REST API, which serves as a standard interface for interacting with various models.

**Why run a model locally instead of calling a hosted API (e.g. OpenAI, Anthropic)?** Running locally means your data never leaves your machine (privacy), there's no per-call cost, and it works offline — at the cost of needing enough local compute/memory, and usually a smaller/weaker model than the largest hosted ones.


In [1]:
%%capture
import sys

# ---------------------------------------------------------------------------
# Environment detection
# ---------------------------------------------------------------------------
# We check whether we're running in Google Colab. This lets a single notebook
# work both in Colab (where we install everything into the VM fresh each
# time) and in the course devcontainer (where dependencies are already
# installed from requirements.txt -- see README.md -- so we skip re-installing them).
#
# We'll reuse IN_COLAB again later in this notebook to decide where to store
# downloaded data, so it's defined here rather than inside the `if` block.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab environment detected. Installing dependencies...")

    # install ollama (version pinned via OLLAMA_VERSION)
    !sudo apt update
    !sudo apt install -y pciutils zstd
    !curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.34.0 sh

    # Python dependencies used later in this notebook:
    #   openai       -> for setting up LLM api client
    #   instructor   -> for handling structured outputs
    #   pydantic     -> for defining the structured output schemas
    #   transformers -> from Hugging Face; for model/tokenizer operations
    %pip install openai==3.3.0 instructor==1.17.0 pydantic==2.14.3 transformers==5.12.1

Ollama typically runs as a service in the background, parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks sequentially, we need a workaround such that the model is available for future cells. To do this, we will create new `thread` and use Python's `subprocess` module to serve the model.

The `thread` ensures that the process runs in parallel to the notebook cells. `subprocess` allows you to run other programs or commands from your Python code.
In this case, it's the command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  # Use subprocess.Popen to start 'ollama serve' in a non-blocking way.
  # This means the Python script doesn't wait for 'ollama serve' to finish.
  subprocess.Popen(["ollama", "serve"])

if IN_COLAB:
    # Create a new thread targeting the run_ollama_serve function.
    # This allows 'ollama serve' to run in the background, separate from the main notebook process.
    thread = threading.Thread(target=run_ollama_serve)
    # Start the thread. This begins the execution of run_ollama_serve concurrently.
    thread.start()
    # Pause the main thread for 5 seconds to give Ollama time to initialize fully.
    time.sleep(5)


To download a model, use `ollama pull`. For the full list of models available, check https://ollama.com/library

Model names typically follow the pattern `family:size` (e.g. `qwen2.5:1.5b`), where the size refers to the number of parameters (here, 1.5 billion). Larger models are generally more capable but slower and more memory-hungry — an important trade-off when running locally rather than on a GPU cluster.


In [3]:
# We default to a small model (1.5 billion parameters) so it runs quickly on modest, hardware (e.g. a laptop or a Colab free-tier instance).
# Feel free to swap in a different model — just make sure your machine has enough RAM, e.g.
# MODEL = 'gemma4:e4b'
# !ollama pull {MODEL}

MODEL = 'qwen2.5:1.5b'
!ollama pull {MODEL}  # downloads the model weights the first time this runs

#### Comprehension Questions

1. What version of the model are we using here? Hint: inspect the model hash id, here for gemma4 https://ollama.com/library/gemma4/tags or https://ollama.com/library/qwen2.5/tags for qwen2.5.

2. What does the `1.5b` in `qwen2.5:1.5b` refer to? How do you expect a 1.5B-parameter model to compare to a much larger one (e.g. 70B) on the tasks in this notebook?

3. Look up `qwen2.5:1.5b` on the Ollama library page. What is its context window (max input length)? Keep this number in mind — we'll revisit it in the Bonus section.

### Test the API Server

Once `ollama serve` is running (started in the background above), it exposes a REST API on `localhost:11434`. Let's check that it's up and that our model is available before we start sending it prompts.

In [4]:
# Query the /v1/models endpoint directly to confirm the server is running
# and that MODEL has been pulled successfully.
!curl -Ss http://localhost:11434/v1/models

{"object":"list","data":[{"id":"qwen2.5:1.5b","object":"model","created":1783671210,"owned_by":"library"}]}


## Interacting with the LLM

Once the API server is running, we can make requests and get responses.

Here, we'll use `OpenAI` python library, although you could also use any other tool/library that wraps OpenAI-compatible APIs (e.g. `langchain` or `llamaindex` for RAG and agentic workflows, `instructor` for forcing structured outputs, etc.)

In [5]:
from openai import OpenAI

# Initialize the OpenAI client with Ollama backend
client = OpenAI(
    # Specify the base URL for the Ollama API endpoint.
    base_url='http://localhost:11434/v1/',
    # Provide an API key.
    api_key='ollama',  # always required but ignored for Ollama
)

# Create a chat completion request to the LLM
chat_completion = client.chat.completions.create(
    # Define the conversation messages as a list of dictionaries
    messages=[
        {
            'role': 'system', # The 'system' role sets the behavior or persona of the AI
            'content': 'You are a poet. You always respond with a clever rhyme.', # System instructions
        },
        {
            'role': 'user', # The 'user' role contains the actual prompt from the user
            'content': 'Say "hello Tannon"', # User's input
        },
    ],
    # Specify the model to use for the completion (e.g., the one served with Ollama)
    model=MODEL,
)

# Display the entire chat_completion object, which contains the model's response and metadata
chat_completion

ChatCompletion(id='chatcmpl-446', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello, Tannon,\nWhispering winds embrace the trees,\nA journey through old, nostalgic songs.\nMay this beat of your heart forever sing.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1783671330, model='qwen2.5:1.5b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=31, prompt_tokens=32, total_tokens=63, completion_tokens_details=None, prompt_tokens_details=None))

In [6]:
# print out the model's response
print(chat_completion.choices[0].message.content)

Hello, Tannon,
Whispering winds embrace the trees,
A journey through old, nostalgic songs.
May this beat of your heart forever sing.


In [7]:
# print out the model's reasoning if available (not supported by all models)
if hasattr(chat_completion.choices[0].message, 'reasoning'):
    print(chat_completion.choices[0].message.reasoning)
else:
    print(f"Chat completion has no reasoning field: {chat_completion}")

Chat completion has no reasoning field: ChatCompletion(id='chatcmpl-446', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello, Tannon,\nWhispering winds embrace the trees,\nA journey through old, nostalgic songs.\nMay this beat of your heart forever sing.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1783671330, model='qwen2.5:1.5b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=31, prompt_tokens=32, total_tokens=63, completion_tokens_details=None, prompt_tokens_details=None))


#### Comprehension Questions

1. How did the model do on this task? Does it match your expectations?

2. Read through your model's reasoning trace (if it produced one). How did the model's "thought process" evolve? Does it match your expectations?

3. What role does the `system` message play, compared to the `user` message? Try editing or removing the system prompt above and re-run the cell — how does the output change?

## Building Prompts in Python

An LLM prompt is a `string`.

It can be as basic or as detailed as you like.

Below we'll look at how to build these with placeholder variables in order to be able to fill the prompt with inputs from a dataset.

In [8]:
# {text} and {task} are placeholders — we'll fill them in later with .format()
example_prompt = "First read the following document:\n\n{text}\n\nand then {task}"
print(example_prompt)

First read the following document:

{text}

and then {task}


In [9]:
# .format() substitutes the placeholders with concrete values.
# This lets us reuse the same prompt template for many different documents/tasks.
text = "This is my document. It has words in it and says stuff."
task = "answer in 1-2 sentences: what's in the document?"
print(example_prompt.format(text=text, task=task))

First read the following document:

This is my document. It has words in it and says stuff.

and then answer in 1-2 sentences: what's in the document?


In [10]:
def query_llm(
        user_prompt: str,
        system_prompt: str = "You are a helpful assistent.",
        model: str = MODEL
        ) -> str:
    """
    Query the LLM with a fully formed prompt.

    Wraps the OpenAI-compatible chat completion call so the rest of the notebook
    can just call query_llm(...) instead of repeating this boilerplate.

    Args:
        user_prompt: The (already-filled-in) prompt sent as the 'user' message.
        system_prompt: Instructions that set the model's behaviour/persona,
            sent as the 'system' message. Defaults to a generic assistant.
        model: The Ollama model tag to query (defaults to the MODEL set above).

    Returns:
        The model's reply as a plain string, or an empty string if the request failed.
    """

    # initialize client
    client = OpenAI(
        base_url='http://localhost:11434/v1/',
        api_key='ollama',  # required but ignored
    )

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            model=model,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        # Fail gracefully (e.g. server not running, model not pulled) rather than
        # crashing the whole loop when we call this function many times below.
        print(f"Error querying LLM: {e}")
        return ''


## Task 1 — Sentiment Analysis (zero-shot)

In session 3 you trained a classifier and ran a fine-tuned transformer for sentiment — both needed labeled data. Here the LLM does it **zero-shot**: using only a natural-language instruction (no training, no labeled examples).

We'll reuse a few IMDB-style reviews; ideally, we would run these on the same set of test reviews you used in session 3 in order to do a full comparison.

In [11]:
from typing import Optional

reviews = [
    "A stunning, heartfelt film — easily the best I've seen all year.",
    "Painfully dull. I checked my phone more than the screen.",
    "It's not bad, exactly, but I wouldn't watch it again.", # tricky: negation + hedge
    "The effects were great, but the plot made no sense at all.", # mixed
]

# The system prompt constrains the *format* of the answer (one word),
# the user prompt template carries the *task* and gets filled in per-review.
sentiment_system_prompt = "You are a sentiment classifier. Always respond with only one word: POSITIVE or NEGATIVE."
sentiment_user_prompt = "Classify the sentiment of the following review as POSITIVE or NEGATIVE. Review: {text}"

for review in reviews:
    sentiment = query_llm(
        user_prompt=sentiment_user_prompt.format(text=review),
        system_prompt=sentiment_system_prompt,
        model=MODEL,
    )
    # :>9 right-aligns the label in a 9-character-wide column so the reviews line up
    print(f"{sentiment:>9}  |  {review[:60]}...")


 POSITIVE  |  A stunning, heartfelt film — easily the best I've seen all y...
 NEGATIVE  |  Painfully dull. I checked my phone more than the screen....
 NEGATIVE  |  It's not bad, exactly, but I wouldn't watch it again....
 NEGATIVE  |  The effects were great, but the plot made no sense at all....


**👉 Your turn** — feed in two reviews the session-3 models got *wrong* (the misclassified ones you inspected).

Reuse the pattern above: define a small list of reviews, then loop over it calling `query_llm()` with `sentiment_system_prompt` / `sentiment_user_prompt`.

In [12]:
# your code here...
# Tip: reuse `sentiment_system_prompt`, `sentiment_user_prompt`, and `query_llm()` from the cell above.
# e.g.
# misclassified_reviews = [...]
# for review in misclassified_reviews:
#     sentiment = query_llm(user_prompt=sentiment_user_prompt.format(text=review), system_prompt=sentiment_system_prompt)
#     print(f"{sentiment:>9}  |  {review[:60]}...")

#### Comprehension Questions

1. Does the LLM handle the negation/sarcasm cases better?

2. At what cost — how does latency compare to the millisecond sklearn prediction?

3. Run the loop again without changing anything. Do you get exactly the same labels back? What does this tell you about the reproducibility of LLM outputs compared to the session-3 classifier?

## Task 2 — Keyword Extraction

In session 2, we used TF-IDF to identify a document's keywords. In session 3, we employed KeyBERT. Let's see how we can do the same with an LLM.




In [13]:
document = "The city council approved a new cycling infrastructure plan on Tuesday, allocating " \
    "12 million francs over three years to build protected bike lanes, expand the bike-share " \
    "program, and redesign five major intersections for safety. Critics questioned the cost; " \
    "supporters cited a 30% rise in cycling commutes since 2023."


# A free-text prompt: we ask for keywords in natural language, with no constraint
# on the output format (unlike the strict POSITIVE/NEGATIVE prompt in Task 1).
keyword_extraction_prompt = """Read the document and provide a list of 4-6 keywords that describe the content.

Document:

{document}
"""

keywords = query_llm(
    user_prompt=keyword_extraction_prompt.format(document=document),
    model=MODEL,
)
print(keywords)


Here are some 4-6 keywords describing the content of your document:

1. Cycling
2. Infrastructure 
3. Plan
4. Funding (12 million francs over three years)
5. Safety
6. Commuters


#### Comprehension Questions

1. How do these keywords compare to those from the methods we've looked at in session 2 and 3?

2. Compare the output with your neighbour's. What potential issues can you see?

3. Imagine you want to use these results as input to a downstream component in your data processing pipeline. How would you process them further?

4. How would you steer the LLM to generate an output that matches an expected format?

## Getting Structured Outputs

As we just saw, given a task prompt, an LLM can generate its output in arbitray and inconsistent formats, making it challenging to actually use them in a pipeline. Parsing the output strings for the actual content that you're interested can be a nightmare.

For this reason, we want to employ **structured outputs**. Most LLM providers implement this in some way in their APIs. A general, cross-platform tool is the [`Instructor` library](https://github.com/567-labs/instructor).

Instructor wraps the model's API endpoint with the appropriate logic for enforcing a JSON output structure matching the one you specify. Specifically, it:

1. Converts your Pydantic model to a schema the LLM can understand
2. Formats the prompt appropriately for the provider
3. Validates the LLM's response against your model
4. Retries automatically if validation fails
5. Returns a properly typed Pydantic object

As a result, your LLM output is guaranteed to be in the expected format.


In [14]:
import instructor
from pydantic import BaseModel

# Define what you want from the LLM
# For a list of keywords, we'll define our response model as a list of strings
class KeywordsList(BaseModel):
    words: list[str]

# Re-initialize the OpenAI client with instructor to enable structured output
client = instructor.from_provider(
    f"ollama/{MODEL}",
    mode=instructor.Mode.JSON, # specify JSON mode for simplicity and model generality
    )

# Make a chat completion request to the LLM
# The response_model argument tells instructor to parse the LLM's output into the KeywordsList Pydantic model
result = client.chat.completions.create(
    response_model=KeywordsList, # Specify the Pydantic model for the expected output structure
    messages=[
        {
            "role": "user",
            "content": keyword_extraction_prompt.format(document=document)
        }
    ],
)

print(result)

words=['city council', 'new cycling infrastructure plan', 'protected bike lanes', 'bike-share program', 'major intersections', 'safety', 'cycling commutes', 'cost']


In [15]:
# result.words is now a plain Python list[str] (not a string to parse!) —
# this is the benefit of structured outputs: we can use it directly.
for word in result.words:
    print(word)

city council
new cycling infrastructure plan
protected bike lanes
bike-share program
major intersections
safety
cycling commutes
cost


## Task 3 — NER

In session 3 the transformer NER pipeline returned deterministic labeled spans. Here we prompt the LLM for the same entities as **structured JSON**.

As a reminder, Named Entity Recognition (NER) identifies and categorizes spans of text into predefined types — here: persons, organizations, locations, and miscellaneous.

In [16]:
from pydantic import BaseModel
import instructor

# Define Pydantic models for the NER output structure.
# Each field becomes a required key in the JSON the LLM must return.
class NEROutput(BaseModel):
    persons: list[str]
    organizations: list[str]
    locations: list[str]
    misc: list[str]

article = "Anna Keller, a researcher at the University of Zurich, presented her findings " \
    "in Geneva on Tuesday. The study, funded by the Swiss National Science Foundation (SNF) " \
    "with 2.4 million francs, analyzed social media posts from Reddit and X. " \
    "Keller previously worked at ETH and at Google in London."


# NOTE: this is a plain string template (not an f-string!), so {text} stays as a
# literal placeholder until we explicitly fill it in with .format() below — the
# same pattern we used for keyword_extraction_prompt above.
ner_prompt = """Extract all named entities from the text below.
Categorize them into persons, organizations, locations, and miscellaneous.

Text:

{text}
"""

# Re-initialize the OpenAI client with instructor to enable structured output
client = instructor.from_provider(
    f"ollama/{MODEL}",
    mode=instructor.Mode.JSON,
    )

# Make a chat completion request to the LLM
result = client.chat.completions.create(
    response_model=NEROutput,
    messages=[
        {"role": "user", "content": ner_prompt.format(text=article)},
    ],
    model=MODEL,
    temperature=0.0, # set temperature for more determinism
)

# The result is already a Pydantic object, no need for manual JSON parsing
print(result.model_dump_json(indent=2))


{
  "persons": [
    "Anna Keller"
  ],
  "organizations": [
    "University of Zurich",
    "Swiss National Science Foundation (SNF)",
    "ETH"
  ],
  "locations": [
    "Geneva"
  ],
  "misc": []
}


### Comparing zero- to few-shot for NER

So far we've asked the LLM to extract entities using only an instruction (zero-shot). Now let's try **few-shot prompting**: giving the model a handful of worked (input/output) examples directly in the prompt, so it can infer the exact format and labeling conventions we want, in addition to following the instructions.

This is a form of *in-context learning* — the model isn't retrained, but its behaviour is steered purely by what's in the prompt.

In [17]:
import json

# Re-initialize the OpenAI client with instructor to enable structured output
client = instructor.from_provider(
    f"ollama/{MODEL}",
    mode=instructor.Mode.JSON,
    )

# Define examples for few-shot prompting.
# Each example pairs an input text with the exact JSON output we'd like for it —
# these will be shown to the model as demonstrations of the task.
ner_examples = [
    {
        "text": "Elon Musk, CEO of Tesla and SpaceX, visited Berlin, Germany, to discuss new factory plans with local officials.",
        "output": {
            "persons": ["Elon Musk"],
            "organizations": ["Tesla", "SpaceX"],
            "locations": ["Berlin", "Germany"],
            "misc": ["CEO"]
        }
    },
    {
        "text": "Dr. Jane Goodall, a famous primatologist, founded the Jane Goodall Institute in 1977. She conducted her groundbreaking research in Gombe Stream National Park, Tanzania.",
        "output": {
            "persons": ["Dr. Jane Goodall"],
            "organizations": ["Jane Goodall Institute"],
            "locations": ["Gombe Stream National Park", "Tanzania"],
            "misc": ["primatologist"]
        }
    },
    {
        "text": "The United Nations announced a new climate initiative at its headquarters in New York City. Representatives from Google and Microsoft were present.",
        "output": {
            "persons": [],
            "organizations": ["United Nations", "Google", "Microsoft"],
            "locations": ["New York City"],
            "misc": ["climate initiative"]
        }
    }
]

# Build a single string that concatenates all examples as
# "Input: ... \n Output: ```json...```" blocks. This string gets inserted
# into the system prompt below, so the model sees all examples before the real task.
example_string = ''
for example in ner_examples:
    example_string += f"Input:\n\n{example['text']}\n\nOutput:\n\n```\n{json.dumps(example['output'])}\n```\n\n"

print(example_string)

Input:

Elon Musk, CEO of Tesla and SpaceX, visited Berlin, Germany, to discuss new factory plans with local officials.

Output:

```
{"persons": ["Elon Musk"], "organizations": ["Tesla", "SpaceX"], "locations": ["Berlin", "Germany"], "misc": ["CEO"]}
```

Input:

Dr. Jane Goodall, a famous primatologist, founded the Jane Goodall Institute in 1977. She conducted her groundbreaking research in Gombe Stream National Park, Tanzania.

Output:

```
{"persons": ["Dr. Jane Goodall"], "organizations": ["Jane Goodall Institute"], "locations": ["Gombe Stream National Park", "Tanzania"], "misc": ["primatologist"]}
```

Input:

The United Nations announced a new climate initiative at its headquarters in New York City. Representatives from Google and Microsoft were present.

Output:

```
{"persons": [], "organizations": ["United Nations", "Google", "Microsoft"], "locations": ["New York City"], "misc": ["climate initiative"]}
```




In [18]:
# {examples} will be filled with the example_string built above;
# the model sees the instructions + worked examples together as its system message.
few_shot_system_message = """You are an expert at Named Entity Recognition. Extract persons, organizations, locations, and miscellaneous entities from the text in JSON format according to the following examples.

{examples}

"""

few_shot_prompt = """Input:

{text}
"""


# Add the actual article to be processed
few_shot_messages = [
    {
        "role": "system",
        "content": few_shot_system_message.format(examples=str(example_string))
    },
    {
        "role": "user",
        "content": few_shot_prompt.format(text=article)
    }
]

result = client.chat.completions.create(
    response_model=NEROutput,
    messages=few_shot_messages,
    model=MODEL,
    temperature=0.0,
)

print(result.model_dump_json(indent=2))

{
  "persons": [
    "Anna Keller"
  ],
  "organizations": [
    "University of Zurich",
    "ETH"
  ],
  "locations": [
    "Geneva"
  ],
  "misc": [
    "Swiss National Science Foundation (SNF)"
  ]
}


#### Comprehension Questions

1. Compare the results for the few-shot and zero-shot settings. How did we do?

2. What are some potential explanations for what you see in the few-shot results?

3. What are the trade-offs for relying on few-shot prompting? Hint: https://docs.ollama.com/context-length

**👉 Your turn** — paste in the `failure_cases` you collected in session 3 and run the NER inference task with the LLM. Compare zero- and few-shot settings and answer the following questions:

1. Compare the NER results above to those you got in session 3. Does the LLM handle any entities that the dedicated Transfomer model missed?

2. Does it produce any problematic annotations?

In [19]:
# your code here...
# Tip: reuse `NEROutput`, `ner_prompt` (zero-shot) and the `few_shot_messages` pattern
# (few-shot) from the cells above, substituting `article` with your own failure-case text(s).

## Bonus: Sanity checking prompt lengths

A common failure point when working with LLMs, especially locally-served models with Ollama is handling input lengths and silent truncation.

If a prompt is too long, the input can be truncated.

As a result, your actual input text, that comes at the end of your prompt, could be compromised!

Therefore, it pays to check these in advance.

The function below demonstrates how to do this with an open-weight model available in Hugging Face — recall the context-window question from the Setup section's Comprehension Questions.

In [20]:
from transformers import AutoTokenizer

def count_prompt_tokens(model_name: str, messages: list[dict]) -> tuple[str, int]:
    """
    Computes the number of input tokens for a given Hugging Face model
    and a list of messages using the model's tokenizer. For OpenAI models, use `tiktoken` instead.

    Args:
        model_name (str): The name of the Hugging Face model (must match a tokenizer on the HF Hub).
        messages (list[dict]): A list of message dictionaries, each with 'role' and 'content' keys —
            e.g. the same `messages` list you'd pass to the chat completion API.

    Returns:
        tuple[str, int]: The fully rendered prompt (with chat template applied), and its token count.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Error loading tokenizer for model {model_name}: {e}")
        return -1 # Indicate an error

    # Depending on the model, the prompt will be constructed with the appropriate chat template
    # This ensures accurate counts with additional markup/structure tags
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    return prompt, len(tokenizer.encode(prompt))

# Reuse the few-shot messages built for Task 3 to see how much the worked
# examples inflate the prompt length compared to the zero-shot version.
full_prompt, num_tokens = count_prompt_tokens("Qwen/Qwen2.5-1.5B", few_shot_messages)
print(f"** Number of input tokens: {num_tokens} **\n\n")
print(full_prompt)


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

** Number of input tokens: 559 **


<|im_start|>system
You are an expert at Named Entity Recognition. Extract persons, organizations, locations, and miscellaneous entities from the text in JSON format according to the following examples.

Input:

Elon Musk, CEO of Tesla and SpaceX, visited Berlin, Germany, to discuss new factory plans with local officials.

Output:

```
{"persons": ["Elon Musk"], "organizations": ["Tesla", "SpaceX"], "locations": ["Berlin", "Germany"], "misc": ["CEO"]}
```

Input:

Dr. Jane Goodall, a famous primatologist, founded the Jane Goodall Institute in 1977. She conducted her groundbreaking research in Gombe Stream National Park, Tanzania.

Output:

```
{"persons": ["Dr. Jane Goodall"], "organizations": ["Jane Goodall Institute"], "locations": ["Gombe Stream National Park", "Tanzania"], "misc": ["primatologist"]}
```

Input:

The United Nations announced a new climate initiative at its headquarters in New York City. Representatives from Google and Microsoft wer

## Discussion

Review the outputs from this notebook and fill in the table below. These points will be used to motivate a class discussion.

| Topic | What did you observe? |
|---|---|
| Hallucination (invented entities / answers) | |
| Reproducibility (two runs differ) | |
| Output structure (output parse failures) | |
| Cost & latency (vs. a small dedicated model) | |
| Privacy (where did your data go) | |
| Context lengths (how long are your prompts/inputs) | |